# MLP для класифікації сортів рису

Датасет: **Rice (Cammeo and Osmancik)**.

Задача: навчити MLP-модель у PyTorch, яка за 7 числовими ознаками зерна визначає сорт рису: `Cammeo` або `Osmancik`.

## Опис датасету

У датасеті 3810 зерен рису двох сортів. Ознаки були отримані після обробки зображень зерен, але тут ми працюємо вже з готовою табличною версією.

| Колонка | Зміст |
|---|---|
| `Area` | площа зерна |
| `Perimeter` | периметр зерна |
| `Major_Axis_Length` | довжина головної осі |
| `Minor_Axis_Length` | довжина малої осі |
| `Eccentricity` | витягнутість форми |
| `Convex_Area` | площа опуклої оболонки |
| `Extent` | співвідношення площі зерна до bounding box |
| `Class` | сорт рису: `Cammeo` або `Osmancik` |

План:

1. завантажити CSV;
2. перевірити структуру даних;
3. підготувати `X` і `y`;
4. зробити train/test split і масштабування;
5. перетворити дані на PyTorch-тензори;
6. побудувати MLP через `nn.Sequential`;
7. написати train loop без батчів;
8. оцінити якість моделі.

## 1. Імпорти та налаштування

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

import torch
from torch import nn

RANDOM_STATE = 42
TEST_SIZE = 0.2
N_EPOCHS = 100
LEARNING_RATE = 0.001

np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

print('PyTorch version:', torch.__version__)

## 2. Завантаження даних

In [ ]:
rice = pd.read_csv('data/rice_cammeo_osmancik.csv')
print('Shape:', rice.shape)
display(rice.head())

In [ ]:
# TODO:
# 1. Подивіться info() для датасету.
# 2. Подивіться describe().T для числових ознак.
# 3. Перевірте баланс класів у колонці Class.

# Ваш код нижче:

## 3. Підготовка X та y

Цільову змінну потрібно перетворити на числа:

```text
Cammeo -> 0
Osmancik -> 1
```

In [ ]:
feature_cols = [col for col in rice.columns if col != 'Class']
class_to_id = {'Cammeo': 0, 'Osmancik': 1}
id_to_class = {value: key for key, value in class_to_id.items()}

# TODO:
# 1. Створіть X з числових колонок feature_cols.
# 2. Створіть y з колонки Class через map(class_to_id).
# 3. Переконайтеся, що X має dtype np.float32, а y має dtype np.int64.

# X = ...
# y = ...

# print('X shape:', X.shape)
# print('y shape:', y.shape)
# print('Class mapping:', class_to_id)

## 4. Train/test split і масштабування

Для MLP важливо масштабувати числові ознаки. `StandardScaler` навчаємо тільки на `X_train`, а потім застосовуємо до `X_test`.

In [ ]:
# TODO:
# 1. Зробіть train_test_split з test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y.
# 2. Створіть StandardScaler.
# 3. Навчіть scaler на X_train і трансформуйте X_train.
# 4. Трансформуйте X_test тим самим scaler.

# X_train, X_test, y_train, y_test = train_test_split(...)
# scaler = StandardScaler()
# X_train_scaled = ...
# X_test_scaled = ...

# print('Train shape:', X_train_scaled.shape)
# print('Test shape:', X_test_scaled.shape)

## 5. PyTorch-тензори

Для `CrossEntropyLoss`:

- `X` має бути `torch.float32`;
- `y` має бути `torch.long`.

In [ ]:
# TODO:
# 1. Перетворіть train/test arrays на torch tensors.
# 2. Перевірте форми створених тензорів.

# X_train_tensor = ...
# y_train_tensor = ...
# X_test_tensor = ...
# y_test_tensor = ...

# print('X_train_tensor:', X_train_tensor.shape)
# print('y_train_tensor:', y_train_tensor.shape)
# print('X_test_tensor:', X_test_tensor.shape)
# print('y_test_tensor:', y_test_tensor.shape)

## 6. MLP через nn.Sequential

Базова архітектура:

```text
Linear(7 -> 16)
ReLU
Linear(16 -> 8)
ReLU
Linear(8 -> 2)
```

Фінальний шар має 2 виходи, бо у нас 2 класи.

In [ ]:
# TODO:
# 1. Визначте input_dim.
# 2. Створіть model через nn.Sequential.
# 3. Створіть loss_fn = nn.CrossEntropyLoss().
# 4. Створіть optimizer = torch.optim.Adam(...).

# input_dim = ...
# model = nn.Sequential(...)
# loss_fn = ...
# optimizer = ...

# model

## 7. Train loop без батчів

У кожній епосі:

1. переведіть модель у train mode;
2. передайте в модель весь `X_train_tensor`;
3. порахуйте loss для всього `y_train_tensor`;
4. зробіть `zero_grad`, `backward`, `step`;
5. збережіть train loss за епоху.

In [ ]:
train_losses = []

# TODO: реалізуйте train loop на N_EPOCHS епох.

# for epoch in range(N_EPOCHS):
#     model.train()
#
#     logits = ...
#     loss = ...
#
#     optimizer.zero_grad()
#     loss.backward()
#     optimizer.step()
#
#     train_losses.append(loss.item())
#
#     if (epoch + 1) % 20 == 0:
#         print(f'Epoch {epoch + 1:3d}/{N_EPOCHS}, train loss: {loss.item():.4f}')

In [ ]:
# TODO: побудуйте графік train_losses.

# plt.figure(figsize=(7, 4))
# plt.plot(train_losses)
# plt.xlabel('Epoch')
# plt.ylabel('Train loss')
# plt.title('MLP training loss')
# plt.grid(alpha=0.3)
# plt.show()

## 8. Оцінка якості

На test set порахуйте:

- `accuracy`;
- `f1_score`;
- `confusion_matrix`.

In [ ]:
def predict(model, X_tensor):
    model.eval()

    with torch.no_grad():
        # TODO: отримайте logits і predictions.
        # logits = ...
        # preds = ...
        pass

    # TODO: поверніть predictions як numpy array.
    # return preds.cpu().numpy()

In [ ]:
# TODO:
# 1. Отримайте y_pred через predict(model, X_test_tensor).
# 2. Підготуйте y_true з y_test_tensor.
# 3. Порахуйте accuracy, F1 і confusion matrix.
# 4. Виведіть метрики.

# y_pred = ...
# y_true = ...
# accuracy = ...
# f1 = ...
# cm = ...

# print(f'Accuracy: {accuracy:.4f}')
# print(f'F1 score: {f1:.4f}')
# print(cm)

In [ ]:
# TODO: побудуйте confusion matrix.

# disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[id_to_class[0], id_to_class[1]])
# disp.plot(cmap='Blues')
# plt.title('Rice classification confusion matrix')
# plt.show()

## 9. Додатковий експеримент

Оберіть 1-2 зміни й порівняйте результат з базовою моделлю:

- `Tanh` або `LeakyReLU` замість `ReLU`;
- `Dropout(p=0.2)`;
- `BatchNorm1d`;
- `weight_decay=1e-4` в `Adam`.

Заповніть таблицю:

| Модель | Accuracy | F1 |
|---|---:|---:|
| Базова MLP | ... | ... |
| MLP + зміна | ... | ... |

In [ ]:
# TODO: проведіть додатковий експеримент і створіть таблицю результатів.

## Висновок

Напишіть 2-3 речення:

- чи навчилася модель розрізняти два сорти рису;
- яка якість на test set;
- чи видно проблему перенавчання за графіком loss;
- чи допомогла ваша додаткова зміна.